In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
np.random.seed(42)
n = 5000
categories = ['Électronique', 'Vêtements', 'Alimentation', 'Livres', 'Maison & Jardin', 'Sport']
pays = ['France', 'Allemagne', 'Espagne', 'Italie', 'Royaume-Uni', 'USA', 'Canada', 'Japon', 'Brésil', 'Australie']
segments = ['Grand Public', 'PME', 'Grand Compte']
canaux = ['Web', 'Mobile', 'Revendeur']
start_date = datetime(2023, 1, 1)
data = {
'order_id': [f'ORD-{i:05d}' for i in range(1, n+1)],
'date': [start_date + timedelta(days=np.random.randint(0, 365)) for _ in range(n)],
'categorie': np.random.choice(categories, n, p=[0.25, 0.20, 0.15, 0.10, 0.15, 0.15]),
'pays': np.random.choice(pays, n, p=[0.18, 0.15, 0.10, 0.10, 0.12, 0.15, 0.07, 0.06, 0.04, 0.03]),
'segment_client': np.random.choice(segments, n, p=[0.60, 0.30, 0.10]),
'canal_vente': np.random.choice(canaux, n, p=[0.50, 0.35, 0.15]),
'quantite': np.random.randint(1, 20, n),
'prix_unitaire': np.round(np.random.exponential(scale=80, size=n) + 5, 2),
'remise': np.random.choice([0, 0.05, 0.10, 0.15, 0.20, 0.25], n, p=[0.40, 0.20, 0.15, 0.12, 0.08, 0.05]),
'frais_livraison': np.round(np.random.uniform(0, 30, n), 2),
'satisfaction_client': np.random.choice([1, 2, 3, 4, 5], n, p=[0.05, 0.10, 0.20, 0.40, 0.25]),
'retour': np.random.choice([True, False], n, p=[0.08, 0.92]),
'note_produit': np.round(np.random.uniform(2.5, 5.0, n), 1),
}
# Injection de valeurs manquantes
df = pd.DataFrame(data)
df.loc[np.random.choice(df.index, 150, replace=False), 'satisfaction_client'] = np.nan
df.loc[np.random.choice(df.index, 80, replace=False), 'frais_livraison'] = np.nan
df.loc[np.random.choice(df.index, 50, replace=False), 'note_produit'] = np.nan
# Injection d'anomalies
df.loc[np.random.choice(df.index, 30, replace=False), 'prix_unitaire'] = -1
df.loc[np.random.choice(df.index, 20, replace=False), 'quantite'] = 0

In [2]:
df.head()

,order_id,date,categorie,pays,segment_client,canal_vente,quantite,prix_unitaire,remise,frais_livraison,satisfaction_client,retour,note_produit
0,ORD-00001,2023-04-13,Électronique,Royaume-Uni,Grand Public,Mobile,3,124.02,0.10,18.40,4.0,False,4.8
1,ORD-00002,2023-12-15,Électronique,Allemagne,Grand Public,Mobile,12,33.29,0.05,19.75,3.0,False,4.6
2,ORD-00003,2023-09-28,Vêtements,France,Grand Public,Web,15,11.20,0.00,0.78,4.0,False,3.7
3,ORD-00004,2023-04-17,Vêtements,Japon,PME,Revendeur,7,20.54,0.05,16.21,5.0,False,3.6
4,ORD-00005,2023-03-13,Livres,Italie,PME,Web,18,12.80,0.10,23.55,4.0,False,3.9


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 13 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   order_id             5000 non-null   object        
 1   date                 5000 non-null   datetime64[ns]
 2   categorie            5000 non-null   object        
 3   pays                 5000 non-null   object        
 4   segment_client       5000 non-null   object        
 5   canal_vente          5000 non-null   object        
 6   quantite             5000 non-null   int64         
 7   prix_unitaire        5000 non-null   float64       
 8   remise               5000 non-null   float64       
 9   frais_livraison      4920 non-null   float64       
 10  satisfaction_client  4850 non-null   float64       
 11  retour               5000 non-null   bool          
 12  note_produit         4950 non-null   float64       
dtypes: bool(1), datetime64[ns](1), fl

In [4]:
df.describe()

,date,quantite,prix_unitaire,remise,frais_livraison,satisfaction_client,note_produit
count,5000,5000.000000,5000.000000,5000.000000,4920.000000,4850.000000,4950.000000
mean,2023-07-01 12:02:35.520000,10.042800,83.364264,0.070400,15.013705,3.692577,3.756889
min,2023-01-01 00:00:00,0.000000,-1.000000,0.000000,0.000000,1.000000,2.500000
25%,2023-04-03 00:00:00,5.000000,26.850000,0.000000,7.540000,3.000000,3.100000
50%,2023-06-28 00:00:00,10.000000,58.965000,0.050000,14.945000,4.000000,3.800000
75%,2023-10-01 00:00:00,15.000000,113.860000,0.100000,22.692500,5.000000,4.400000
max,2023-12-31 00:00:00,19.000000,662.990000,0.250000,30.000000,5.000000,5.000000
std,NaN,5.443978,79.486551,0.076459,8.714973,1.107599,0.721788


In [5]:
df.isnull().sum()

,0
order_id,0
date,0
categorie,0
pays,0
segment_client,0
canal_vente,0
quantite,0
prix_unitaire,0
remise,0
frais_livraison,80


In [6]:
df = df[df['prix_unitaire'] > 0]
print("Lignes restantes:", len(df))

Lignes restantes: 4970


In [7]:
df = df[df['quantite'] != 0]
print("Lignes restantes:", len(df))

Lignes restantes: 4950


In [8]:
median_frais = df['frais_livraison'].median()
df['frais_livraison'] = df['frais_livraison'].fillna(median_frais)
print("Médiane utilisée:", median_frais)

Médiane utilisée: 14.945


In [9]:
mode_satisfaction = df['satisfaction_client'].mode()[0]
df['satisfaction_client'] = df['satisfaction_client'].fillna(mode_satisfaction)
print("Mode utilisé:", mode_satisfaction)

Mode utilisé: 4.0


In [10]:
df['note_produit'] = df['note_produit'].fillna(
    df.groupby('categorie')['note_produit'].transform('mean')
)
print("NaN restants dans note_produit:", df['note_produit'].isnull().sum())

NaN restants dans note_produit: 0


In [11]:
df['chiffre_affaires'] = df['quantite'] * df['prix_unitaire'] * (1 - df['remise'])
df['chiffre_affaires'].head()

,chiffre_affaires
0,334.854
1,379.506
2,168.000
3,136.591
4,207.360


In [12]:
df['marge'] = df['chiffre_affaires'] - (df['frais_livraison'] * df['quantite'])
df['marge'].head()

,marge
0,279.654
1,142.506
2,156.300
3,23.121
4,-216.540


In [13]:
df['date'] = pd.to_datetime(df['date'])
df['mois'] = df['date'].dt.strftime('%B')  # ex: 'January'
df['mois'].value_counts()

,count
mois,
May,437
August,437
June,436
April,428
December,425
January,414
October,413
November,407
March,401


In [14]:
df['trimestre'] = df['date'].dt.quarter.map({1: 'Q1', 2: 'Q2', 3: 'Q3', 4: 'Q4'})
df['trimestre'].value_counts()

,count
trimestre,
Q2,1301
Q4,1245
Q1,1203
Q3,1201


In [15]:
df['panier_moyen'] = df['chiffre_affaires'] / df['quantite']
df['panier_moyen'].head()

,panier_moyen
0,111.6180
1,31.6255
2,11.2000
3,19.5130
4,11.5200


In [16]:
df[['chiffre_affaires', 'marge', 'mois', 'trimestre', 'panier_moyen']].head(10)

,chiffre_affaires,marge,mois,trimestre,panier_moyen
0,334.8540,279.6540,April,Q2,111.6180
1,379.5060,142.5060,December,Q4,31.6255
2,168.0000,156.3000,September,Q3,11.2000
3,136.5910,23.1210,April,Q2,19.5130
4,207.3600,-216.5400,March,Q1,11.5200
5,409.6500,378.6000,July,Q3,27.3100
6,412.1480,312.9480,January,Q1,51.5185
7,1490.6025,1402.4025,April,Q2,165.6225
8,2542.0500,2130.9000,May,Q2,169.4700
9,1617.8880,1528.7680,August,Q3,101.1180


In [17]:
result_15 = df.groupby('categorie').agg(
    CA_total=('chiffre_affaires', 'sum'),
    marge_totale=('marge', 'sum'),
    nb_commandes=('order_id', 'count')
).sort_values('CA_total', ascending=False)

print(result_15)

                    CA_total  marge_totale  nb_commandes
categorie                                               
Électronique     992615.1585   801312.9485          1286
Vêtements        817651.0650   660935.7450          1031
Alimentation     611277.4485   494609.7285           758
Sport            549097.8230   439881.6780           710
Maison & Jardin  541707.2310   436297.9210           690
Livres           361694.9010   289942.5010           475


In [18]:
ca_pays = df.groupby('pays')['chiffre_affaires'].sum().sort_values(ascending=False)

print("=== Top 3 pays ===")
print(ca_pays.head(3))
print("\n=== Bottom 3 pays ===")
print(ca_pays.tail(3))

=== Top 3 pays ===
pays
France       711987.0340
USA          587601.0475
Allemagne    577907.7460
Name: chiffre_affaires, dtype: float64

=== Bottom 3 pays ===
pays
Japon        207980.2615
Brésil       166522.5910
Australie    129931.5700
Name: chiffre_affaires, dtype: float64


In [19]:
result_17 = df.groupby('canal_vente').agg(
    remise_moyenne=('remise', 'mean'),
    taux_retour=('retour', 'mean')
)
result_17['taux_retour'] = (result_17['taux_retour'] * 100).round(2)
result_17['remise_moyenne'] = result_17['remise_moyenne'].round(4)

print(result_17)

             remise_moyenne  taux_retour
canal_vente                             
Mobile               0.0696         8.91
Revendeur            0.0635         7.79
Web                  0.0732         7.98


In [20]:
pivot_18 = df.pivot_table(
    values='satisfaction_client',
    index='segment_client',
    columns='categorie',
    aggfunc='mean'
).round(2)

print(pivot_18)

categorie       Alimentation  Livres  Maison & Jardin  Sport  Vêtements  \
segment_client                                                            
Grand Compte            3.76    3.71             3.49   3.63       3.75   
Grand Public            3.68    3.71             3.76   3.66       3.65   
PME                     3.81    3.61             3.78   3.72       3.65   

categorie       Électronique  
segment_client                
Grand Compte            3.75  
Grand Public            3.71  
PME                     3.72  


In [22]:
ca_mois = df.groupby('mois')['chiffre_affaires'].sum()

mois_fort = ca_mois.idxmax()
mois_faible = ca_mois.idxmin()
variation = round((ca_mois.max() - ca_mois.min()) / ca_mois.min() * 100, 2)

print(f"Mois le plus fort  : {mois_fort}  → {ca_mois.max():,.2f} €")
print(f"Mois le plus faible: {mois_faible}  → {ca_mois.min():,.2f} €")
print(f"Variation          : {variation} %")

Mois le plus fort  : August  → 365,487.59 €
Mois le plus faible: July  → 274,354.87 €
Variation          : 33.22 %


In [23]:
ca_trim = df.groupby('trimestre')['chiffre_affaires'].sum().sort_index()

ca_trim_df = ca_trim.reset_index()
ca_trim_df['croissance_%'] = ca_trim_df['chiffre_affaires'].pct_change() * 100
ca_trim_df['croissance_%'] = ca_trim_df['croissance_%'].round(2)

print(ca_trim_df)

  trimestre  chiffre_affaires  croissance_%
0        Q1      9.254854e+05           NaN
1        Q2      1.031368e+06         11.44
2        Q3      9.362517e+05         -9.22
3        Q4      9.809384e+05          4.77


In [24]:
df['jour_semaine'] = df['date'].dt.day_name()

commandes_jour = df.groupby('jour_semaine')['order_id'].count().sort_values(ascending=False)

print("Commandes par jour de la semaine :")
print(commandes_jour)
print(f"\nJour avec le plus de commandes : {commandes_jour.idxmax()} ({commandes_jour.max()} commandes)")

Commandes par jour de la semaine :
jour_semaine
Friday       730
Saturday     725
Thursday     722
Sunday       711
Monday       700
Tuesday      689
Wednesday    673
Name: order_id, dtype: int64

Jour avec le plus de commandes : Friday (730 commandes)


In [25]:
filtre_22 = df[(df['remise'] >= 0.20) & (df['chiffre_affaires'] > 500)]

print(f"Nombre de commandes correspondantes : {len(filtre_22)}")
print(f"Pourcentage du total : {len(filtre_22)/len(df)*100:.2f} %")
print(filtre_22[['order_id', 'categorie', 'remise', 'chiffre_affaires']].head(10))

Nombre de commandes correspondantes : 256
Pourcentage du total : 5.17 %
      order_id        categorie  remise  chiffre_affaires
27   ORD-00028  Maison & Jardin    0.20         1090.4080
29   ORD-00030        Vêtements    0.20         1589.2800
30   ORD-00031     Alimentation    0.25          592.2000
43   ORD-00044     Alimentation    0.25         2766.3675
50   ORD-00051        Vêtements    0.20         4177.7920
72   ORD-00073  Maison & Jardin    0.25          577.2600
109  ORD-00110     Alimentation    0.20          773.8640
112  ORD-00113     Alimentation    0.25          973.4175
138  ORD-00139  Maison & Jardin    0.25          797.4300
141  ORD-00142        Vêtements    0.20          638.7920


In [26]:
france_23 = df[df['pays'] == 'France'].nlargest(10, 'panier_moyen')

print(france_23[['order_id', 'date', 'categorie', 'quantite', 'prix_unitaire', 'remise', 'panier_moyen']])

       order_id       date        categorie  quantite  prix_unitaire  remise  \
204   ORD-00205 2023-07-06  Maison & Jardin         3         552.49    0.00   
3449  ORD-03450 2023-02-20  Maison & Jardin         6         653.20    0.20   
2055  ORD-02056 2023-07-12           Livres         5         662.99    0.25   
418   ORD-00419 2023-09-19            Sport        10         403.76    0.00   
1581  ORD-01582 2023-04-30           Livres        11         472.87    0.15   
4281  ORD-04282 2023-07-16     Alimentation        14         372.48    0.00   
1768  ORD-01769 2023-02-19  Maison & Jardin        19         388.86    0.05   
3453  ORD-03454 2023-10-27     Alimentation         6         448.21    0.20   
4707  ORD-04708 2023-05-04           Livres         9         407.09    0.15   
3321  ORD-03322 2023-02-20        Vêtements         2         330.57    0.00   

      panier_moyen  
204       552.4900  
3449      522.5600  
2055      497.2425  
418       403.7600  
1581      401.

In [27]:
moyenne_ca = df['chiffre_affaires'].mean()

retours_24 = df[(df['retour'] == True) & (df['chiffre_affaires'] > moyenne_ca)]

ca_retours = retours_24['chiffre_affaires'].sum()
ca_total = df['chiffre_affaires'].sum()
poids = (ca_retours / ca_total * 100).round(2)

print(f"Moyenne CA globale          : {moyenne_ca:.2f} €")
print(f"Nb commandes retournées > moyenne : {len(retours_24)}")
print(f"CA de ces retours           : {ca_retours:,.2f} €")
print(f"Poids en % du CA total      : {poids} %")

Moyenne CA globale          : 782.64 €
Nb commandes retournées > moyenne : 135
CA de ces retours           : 237,328.80 €
Poids en % du CA total      : 6.13 %
